In [ ]:
# ============================================================
#  环境配置
#  - Colab 用户：取消注释下方 Colab 区块
#  - 本地 Jupyter 用户：直接运行 Local 区块
# ============================================================

# ── Colab 环境（取消注释后运行） ──
# !pip install torch torchvision -q
# !pip install transformers -q
# !pip install matplotlib numpy scipy Pillow requests -q

# ── 本地 Jupyter 环境 ──
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch', 'torchvision', 'transformers', 'matplotlib', 'numpy', 'scipy', 'Pillow', 'requests']:
    _install(pkg)

In [ ]:
import math
from io import BytesIO

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
import torchvision.ops as ops
import torchvision.transforms as T
from PIL import Image
from scipy.optimize import linear_sum_assignment
from torch import nn

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# DETR 代码实战：学习实现 vs 工程实现

基于论文 *End-to-End Object Detection with Transformers*（Carion et al., ECCV 2020），
用 **目标检测** 任务演示 DETR 的核心思想：`object query + encoder-decoder Transformer + Hungarian matching`。

本 Notebook 采用两条并行路径，并尽量围绕 **同一张 COCO 示例图像** 展开：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 看清 DETR 每个关键模块在做什么 | 掌握工业级预训练 DETR 的使用方式 |
| 实现方式 | `L2: Key module demo`，手写 backbone / PE / encoder / decoder / matching | `E1: Mature library + inference-only`，调用 HuggingFace `transformers` |
| 代码量 | 中等，强调 shape 与公式对应 | 更少，强调 API 与推理流程 |
| 适合场景 | 面试准备、理解机制 | 工程验证、快速落地 |

> 对 DETR 来说，最稳定的教学方式不是“从零训练一个可用检测器”，而是“手写关键模块 + 用预训练模型完成真实推理”。

## 一、论文与任务背景

### 1.1 论文信息

- **论文**：*End-to-End Object Detection with Transformers*
- **作者**：Nicolas Carion, Francisco Massa, Gabriel Synnaeve, Nicolas Usunier, Alexander Kirillov, Sergey Zagoruyko
- **会议**：ECCV 2020
- **任务类型**：`object_detection`

### 1.2 DETR 想解决什么问题

传统检测器通常包含这些步骤：

1. 生成大量候选框 / anchor
2. 用启发式规则分配正负样本
3. 回归边界框
4. 最后再做 NMS 去重

DETR 把它改写成了一个 **集合预测（set prediction）** 问题：

- 模型固定输出 `N` 个 query 对应的检测结果
- 训练时用 **Hungarian matching** 做一对一匹配
- 推理时直接过滤 `no object`，**不需要 NMS**

### 1.3 为什么目标检测任务最能体现 DETR 的核心思想

因为 DETR 的创新不只在 Transformer 本身，还在：

- 2D 位置编码如何进入注意力
- object query 如何代表“我要找一个物体”
- 一对一匹配如何替代 anchor 分配和 NMS

### 1.4 本 Notebook 的可运行性决策

```yaml
model_name: DETR
paper: End-to-End Object Detection with Transformers (Carion et al., 2020)
task_type: object_detection
training_vs_inference_diff: true
learning_path_depth: L2
learning_rationale: 原版 DETR 从零训练成本高，最稳定的学习产物是“关键模块可运行拆解 + matching/loss 演示”
engineering_path_form: E1
engineering_rationale: HuggingFace transformers 已提供成熟的 DETR 预训练推理流程
engineering_action: inference-only
action_rationale: 真实可用的 DETR fine-tune 超出免费 Colab 友好范围，推理更稳定
```

### 1.5 Notebook 覆盖范围

**会覆盖**：

- backbone → 2D positional encoding → encoder → decoder → prediction heads
- Hungarian matching 与 set loss 的最小可运行演示
- 真实预训练 DETR 的推理、后处理、批量推理与 attention 可视化

**不会覆盖**：

- COCO 全量训练
- 分布式训练、auxiliary losses、多尺度训练细节
- Deformable DETR / DINO 的完整实现

## 二、最小必要理论

下面只保留理解代码所必需的公式。

### 2.1 Backbone + 位置编码

设 backbone 输出特征图为：

$$F \in \mathbb{R}^{C \times H' \times W'}$$

再用 `1×1 conv` 投影到 Transformer 维度：

$$X = \text{Proj}(F) \in \mathbb{R}^{d \times H' \times W'}$$

对每个空间位置加入二维位置编码：

$$\text{Input to Encoder} = X + P$$

### 2.2 Encoder / Decoder

自注意力基本形式：

$$\text{Attention}(Q,K,V)=\text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

DETR decoder 的关键是 `N` 个可学习 object queries：

$$Q_{obj} = [q_1, q_2, ..., q_N], \quad q_i \in \mathbb{R}^d$$

它们通过 cross-attention 去“查询”图像特征。

### 2.3 集合预测与 Hungarian Matching

设预测为 $\hat{y}_i=(\hat{c}_i, \hat{b}_i)$，真值为 $y_j=(c_j,b_j)$，代价矩阵为：

$$\text{cost}_{i,j}=\alpha\,\text{cls cost}+\beta\,\text{L1 cost}+\gamma\,\text{GIoU cost}$$

最优匹配：

$$\sigma^*=\arg\min_{\sigma}\sum_j \text{cost}_{\sigma(j),j}$$

### 2.4 训练与推理分离

- **训练**：先 matching，再计算分类损失与框损失
- **推理**：不做 matching，直接保留非 `no object` 的高置信预测

### 2.5 论文组件映射表

| 论文组件 | 学习路径覆盖 | 工程库/API 对应 | 状态 |
|---|---|---|---|
| CNN Backbone + `1×1` 投影 | 手写简化版 backbone | `DetrForObjectDetection.model.backbone` + input projection | Runnable |
| 2D Positional Encoding | 手写 2D sinusoidal PE | 模型内部位置编码模块 | Runnable |
| Transformer Encoder | 手写 encoder block | 模型内部 encoder | Runnable |
| Object Query | 手写 `nn.Embedding(num_queries, d_model)` | `config.num_queries` 对应的 learned queries | Runnable |
| Transformer Decoder | 手写 self-attn + cross-attn | 模型内部 decoder | Runnable |
| 分类头 + 边框头 | 手写 linear / MLP heads | `outputs.logits` 与 `outputs.pred_boxes` | Runnable |
| Hungarian Matching | 手写 cost matrix + `linear_sum_assignment` | 训练内部 matcher | Runnable |
| 分类 + L1 + GIoU 损失 | 手写最小版 loss | 训练内部 criterion | Runnable |
| 推理阶段过滤 `no object` | 手写 softmax + threshold | `post_process_object_detection()` | Runnable |
| 无 NMS 端到端输出 | Markdown 说明 + 工程推理观察 | HF DETR 标准推理流程 | Explain-only |

## 1. 数据准备

为了让学习路径和工程路径尽量对齐，这里都围绕 **同一张 COCO 示例图像** 展开：

- 学习路径：用这张图演示 feature map、query、matching、loss 与推理过滤
- 工程路径：用同一张图跑预训练 `facebook/detr-resnet-50`

另外，学习路径中的真值框只做 **教学演示**，不是 COCO 官方标注。目标是让 matching / loss / inference 代码能跑通。

In [ ]:
IMAGE_URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'
response = requests.get(IMAGE_URL, timeout=30)
response.raise_for_status()
image = Image.open(BytesIO(response.content)).convert('RGB')
ORIG_W, ORIG_H = image.size
print(f'Original image size: {(ORIG_W, ORIG_H)}')

LEARNING_CLASSES = ['cat', 'couch']
LEARNING_LABEL_TO_ID = {name: idx for idx, name in enumerate(LEARNING_CLASSES)}

# 教学演示用近似标注框：两只猫 + 一个沙发
manual_boxes_xyxy = torch.tensor([
    [15.0, 52.0, 315.0, 472.0],   # left cat
    [343.0, 22.0, 640.0, 369.0],  # right cat
    [0.0, 150.0, 640.0, 479.0],   # couch
], dtype=torch.float32)
manual_labels = torch.tensor([
    LEARNING_LABEL_TO_ID['cat'],
    LEARNING_LABEL_TO_ID['cat'],
    LEARNING_LABEL_TO_ID['couch'],
], dtype=torch.long)

plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.title('Raw COCO Sample Image')
plt.axis('off')
plt.show()

In [ ]:
LEARNING_IMAGE_SIZE = 160
learning_transform = T.Compose([
    T.Resize((LEARNING_IMAGE_SIZE, LEARNING_IMAGE_SIZE)),
    T.ToTensor(),
])
learning_image = learning_transform(image).unsqueeze(0).to(device)  # (1, 3, 160, 160)

scale_boxes_xyxy = manual_boxes_xyxy.clone()
scale_boxes_xyxy[:, [0, 2]] = scale_boxes_xyxy[:, [0, 2]] / ORIG_W
scale_boxes_xyxy[:, [1, 3]] = scale_boxes_xyxy[:, [1, 3]] / ORIG_H

cx = (scale_boxes_xyxy[:, 0] + scale_boxes_xyxy[:, 2]) / 2
cy = (scale_boxes_xyxy[:, 1] + scale_boxes_xyxy[:, 3]) / 2
w = scale_boxes_xyxy[:, 2] - scale_boxes_xyxy[:, 0]
h = scale_boxes_xyxy[:, 3] - scale_boxes_xyxy[:, 1]
manual_boxes_cxcywh = torch.stack([cx, cy, w, h], dim=-1)

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.imshow(image)
for label_id, box in zip(manual_labels, manual_boxes_xyxy):
    x1, y1, x2, y2 = box.tolist()
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='lime', facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, max(5, y1 - 8), LEARNING_CLASSES[label_id], color='white', bbox=dict(facecolor='green', alpha=0.7, pad=2))
ax.set_title('Manual Teaching Targets')
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. 共享超参数与工具

先集中管理学习路径要用到的超参数。这里刻意把规模缩小，只追求 **形状正确、逻辑清晰、CPU/GPU 都能跑**。

In [ ]:
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
D_FF = 256
DROPOUT = 0.1
NUM_QUERIES = 12
BACKBONE_CHANNELS = 128
NO_OBJECT_COEF = 0.1
LEARNING_THRESHOLD = 0.35


def box_cxcywh_to_xyxy(boxes):
    cx, cy, w, h = boxes.unbind(-1)
    return torch.stack([cx - 0.5 * w, cy - 0.5 * h, cx + 0.5 * w, cy + 0.5 * h], dim=-1)


print({
    'D_MODEL': D_MODEL,
    'NUM_HEADS': NUM_HEADS,
    'NUM_QUERIES': NUM_QUERIES,
    'DEVICE': str(device),
})

## 3. 学习路径：关键模块拆解（L2）

这一部分不追求“训练出一个高精度检测器”，而是追求：

1. 每个模块都能独立运行
2. 输入输出 shape 清楚
3. 能把论文公式映射到代码
4. 能演示训练阶段的 matching / loss，以及推理阶段的 filtering

In [ ]:
class PositionalEncoding2D(nn.Module):
    def __init__(self, d_model=D_MODEL, temperature=10000):
        super().__init__()
        assert d_model % 2 == 0
        self.d_model = d_model
        self.temperature = temperature

    def forward(self, x):
        # x: (B, d_model, H, W)
        _, _, h, w = x.shape
        half_dim = self.d_model // 2

        y_embed = torch.linspace(0, 1, steps=h, device=x.device)
        x_embed = torch.linspace(0, 1, steps=w, device=x.device)
        dim_t = torch.arange(half_dim, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * torch.div(dim_t, 2, rounding_mode='floor') / half_dim)

        pos_y = y_embed[:, None] / dim_t[None, :]
        pos_x = x_embed[:, None] / dim_t[None, :]

        pos_y = torch.stack((pos_y[:, 0::2].sin(), pos_y[:, 1::2].cos()), dim=-1).flatten(1)
        pos_x = torch.stack((pos_x[:, 0::2].sin(), pos_x[:, 1::2].cos()), dim=-1).flatten(1)

        pos = torch.cat([
            pos_y[:, None, :].expand(h, w, -1),
            pos_x[None, :, :].expand(h, w, -1),
        ], dim=-1)
        return pos.permute(2, 0, 1).unsqueeze(0)

### 3.1 Backbone 与输入投影

DETR 先把图像变成低分辨率特征图，再用 `1×1 conv` 投影到 `d_model` 维。

如果输入是 `(batch, 3, H, W)`，则输出大致变成：

- backbone 后：`(batch, C, H', W')`
- projection 后：`(batch, d_model, H', W')`

这里用一个很小的 CNN 模拟真实的 ResNet backbone。

In [ ]:
class SimpleBackbone(nn.Module):
    def __init__(self, d_model=D_MODEL, hidden_channels=BACKBONE_CHANNELS):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, hidden_channels, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
        )
        self.input_proj = nn.Conv2d(hidden_channels, d_model, kernel_size=1)

    def forward(self, x):
        x = self.features(x)      # (B, hidden_channels, 20, 20)
        x = self.input_proj(x)    # (B, d_model, 20, 20)
        return x

learning_backbone = SimpleBackbone().to(device)
feature_map = learning_backbone(learning_image)
pe = PositionalEncoding2D().to(device)
pos_map = pe(feature_map)

print('learning_image:', tuple(learning_image.shape))
print('feature_map   :', tuple(feature_map.shape))
print('pos_map       :', tuple(pos_map.shape))

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for idx, ax in enumerate(axes):
    channel_id = idx * (D_MODEL // 4)
    ax.imshow(pos_map[0, channel_id].detach().cpu().numpy(), cmap='RdBu_r')
    ax.set_title(f'PE channel {channel_id}')
    ax.axis('off')
plt.tight_layout()
plt.show()

### 3.2 2D Positional Encoding

DETR 不是把图像切成文字 token，而是把 `H'×W'` 的空间特征送进 Transformer。
因此它必须显式告诉模型“这个 token 来自哪个空间位置”。

二维位置编码的直觉是：

- `y` 方向编码负责“上下位置”
- `x` 方向编码负责“左右位置”
- 最后拼接成一个 `d_model` 维向量

这里保留最关键的 shape：

- 输入：`(batch, d_model, H', W')`
- 输出：`(1, d_model, H', W')`

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, pos):
        # src: (B, HW, d_model)
        # pos: (B, HW, d_model)
        q = src + pos
        k = src + pos
        v = src
        attn_out, attn_weights = self.self_attn(q, k, v, need_weights=True)
        src = self.norm1(src + self.dropout(attn_out))
        src = self.norm2(src + self.dropout(self.ffn(src)))
        return src, attn_weights

src_tokens = feature_map.flatten(2).permute(0, 2, 1)           # (B, HW, d_model)
pos_tokens = pos_map.flatten(2).permute(0, 2, 1).expand(src_tokens.size(0), -1, -1)

encoder_block = EncoderBlock().to(device)
memory_tokens, encoder_attn = encoder_block(src_tokens, pos_tokens)
print('src_tokens    :', tuple(src_tokens.shape))
print('memory_tokens :', tuple(memory_tokens.shape))
print('encoder_attn  :', tuple(encoder_attn.shape))

### 3.3 Transformer Encoder

Encoder 把展平后的空间 token 进行全局建模。

如果 `feature_map` 形状是 `(B, d_model, H', W')`，则送入 encoder 前会变成：

$$
(B, d, H', W') \rightarrow (B, H'W', d)
$$

这里用 PyTorch 的 `nn.MultiheadAttention(batch_first=True)` 来写一个最小 encoder block。
注意：DETR 中位置编码主要影响 `Q/K`，而 `V` 仍保留语义特征。

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model=D_MODEL, num_heads=NUM_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, query_pos, memory_pos):
        self_out, self_w = self.self_attn(tgt + query_pos, tgt + query_pos, tgt, need_weights=True)
        tgt = self.norm1(tgt + self.dropout(self_out))

        cross_out, cross_w = self.cross_attn(tgt + query_pos, memory + memory_pos, memory, need_weights=True)
        tgt = self.norm2(tgt + self.dropout(cross_out))

        tgt = self.norm3(tgt + self.dropout(self.ffn(tgt)))
        return tgt, self_w, cross_w

query_embed = nn.Embedding(NUM_QUERIES, D_MODEL).to(device)
query_pos = query_embed.weight.unsqueeze(0).expand(memory_tokens.size(0), -1, -1)  # (B, N, d_model)
tgt = torch.zeros_like(query_pos)                                                    # (B, N, d_model)

decoder_block = DecoderBlock().to(device)
decoder_tokens, self_attn_w, cross_attn_w = decoder_block(tgt, memory_tokens, query_pos, pos_tokens)
print('query_pos      :', tuple(query_pos.shape))
print('decoder_tokens :', tuple(decoder_tokens.shape))
print('cross_attn_w   :', tuple(cross_attn_w.shape))

### 3.4 Object Query + Decoder

这部分是 DETR 最关键的地方。

- `object query` 不是从图像里切出来的 patch
- 它是 **可学习参数**
- 每个 query 表示“我想找一个潜在目标”

decoder 的数据流：

1. query 之间先做 self-attention，避免重复预测
2. 再对 encoder memory 做 cross-attention，去图像特征里“找目标”
3. 最后输出 `N` 个 query 对应的检测表示

对应 shape：

- memory: `(B, HW, d_model)`
- queries: `(B, N, d_model)`
- decoder output: `(B, N, d_model)`

In [ ]:
class PredictionHeads(nn.Module):
    def __init__(self, d_model=D_MODEL, num_classes=len(LEARNING_CLASSES)):
        super().__init__()
        self.class_head = nn.Linear(d_model, num_classes + 1)  # +1 for no object
        self.box_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 4),
        )

    def forward(self, x):
        logits = self.class_head(x)           # (B, N, C+1)
        boxes = self.box_head(x).sigmoid()    # (B, N, 4) in normalized cxcywh
        return logits, boxes

heads = PredictionHeads().to(device)
class_logits, pred_boxes = heads(decoder_tokens)
print('class_logits:', tuple(class_logits.shape))
print('pred_boxes  :', tuple(pred_boxes.shape))
print('box value range:', float(pred_boxes.min()), float(pred_boxes.max()))

### 3.5 Prediction Heads

decoder 输出的每个 query 都会走两条头：

- **分类头**：输出 `num_classes + 1`，最后一个类代表 `no object`
- **边框头**：输出归一化 `cx, cy, w, h`

这是 DETR 和普通分类 Transformer 的本质区别：它不是只输出一个全局类别，而是输出一组对象级结果。

### 3.6 Hungarian Matching、损失函数与训练/推理差异

到这一步，学习路径已经得到了：

- `class_logits`: `(B, N, C+1)`
- `pred_boxes`: `(B, N, 4)`

训练时要做的事情是：

1. 用 Hungarian matching 把 `N` 个 query 和真实目标做一对一匹配
2. 对匹配到的 query 计算分类损失、L1 损失、GIoU 损失
3. 未匹配 query 学习 `no object`

推理时则简单得多：

- softmax
- 去掉 `no object`
- 用阈值过滤

In [ ]:
def generalized_box_iou(boxes1, boxes2):
    return ops.generalized_box_iou(boxes1, boxes2)


@torch.no_grad()
def hungarian_match(pred_logits, pred_boxes, targets):
    batch_indices = []
    probs = pred_logits.softmax(-1)

    for b in range(pred_logits.size(0)):
        tgt_labels = targets[b]['labels'].to(pred_logits.device)
        tgt_boxes = targets[b]['boxes'].to(pred_logits.device)
        if tgt_labels.numel() == 0:
            batch_indices.append((torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)))
            continue

        cls_cost = -probs[b][:, tgt_labels]
        l1_cost = torch.cdist(pred_boxes[b], tgt_boxes, p=1)
        giou_cost = -generalized_box_iou(
            box_cxcywh_to_xyxy(pred_boxes[b]),
            box_cxcywh_to_xyxy(tgt_boxes),
        )
        cost = cls_cost + 5.0 * l1_cost + 2.0 * giou_cost
        row_ind, col_ind = linear_sum_assignment(cost.detach().cpu().numpy())
        batch_indices.append((torch.as_tensor(row_ind, dtype=torch.long), torch.as_tensor(col_ind, dtype=torch.long)))

    return batch_indices


def detr_loss(pred_logits, pred_boxes, targets, matches, eos_coef=NO_OBJECT_COEF):
    num_classes = pred_logits.size(-1) - 1
    no_obj_id = num_classes
    target_classes = torch.full(pred_logits.shape[:2], no_obj_id, dtype=torch.long, device=pred_logits.device)

    matched_pred_boxes = []
    matched_tgt_boxes = []
    for b, (pred_ids, tgt_ids) in enumerate(matches):
        if pred_ids.numel() > 0:
            target_classes[b, pred_ids] = targets[b]['labels'][tgt_ids].to(pred_logits.device)
            matched_pred_boxes.append(pred_boxes[b, pred_ids])
            matched_tgt_boxes.append(targets[b]['boxes'][tgt_ids].to(pred_logits.device))

    class_weight = torch.ones(num_classes + 1, device=pred_logits.device)
    class_weight[no_obj_id] = eos_coef
    cls_loss = F.cross_entropy(pred_logits.flatten(0, 1), target_classes.flatten(), weight=class_weight)

    if matched_pred_boxes:
        matched_pred_boxes = torch.cat(matched_pred_boxes, dim=0)
        matched_tgt_boxes = torch.cat(matched_tgt_boxes, dim=0)
        l1_loss = F.l1_loss(matched_pred_boxes, matched_tgt_boxes)
        giou = generalized_box_iou(
            box_cxcywh_to_xyxy(matched_pred_boxes),
            box_cxcywh_to_xyxy(matched_tgt_boxes),
        )
        giou_loss = (1 - giou.diag()).mean()
    else:
        l1_loss = pred_boxes.sum() * 0.0
        giou_loss = pred_boxes.sum() * 0.0

    total = cls_loss + 5.0 * l1_loss + 2.0 * giou_loss
    return {
        'total': total,
        'cls': cls_loss.detach().item(),
        'l1': l1_loss.detach().item(),
        'giou': giou_loss.detach().item(),
    }

targets = [{'labels': manual_labels, 'boxes': manual_boxes_cxcywh}]
matches = hungarian_match(class_logits, pred_boxes, targets)
loss_dict = detr_loss(class_logits, pred_boxes, targets, matches)
print('matches:', [(p.tolist(), t.tolist()) for p, t in matches])
print('losses :', {k: (v.item() if hasattr(v, 'item') else v) for k, v in loss_dict.items()})

In [ ]:
with torch.no_grad():
    probs = class_logits.softmax(-1)[0]          # (N, C+1)
    scores, labels = probs[:, :-1].max(dim=-1)   # ignore no-object column

selected = scores > LEARNING_THRESHOLD
selected_scores = scores[selected]
selected_labels = labels[selected]
selected_boxes = pred_boxes[0, selected]

print('selected queries:', int(selected.sum().item()))
for idx, (score, label_id, box) in enumerate(zip(selected_scores, selected_labels, selected_boxes)):
    print(idx, LEARNING_CLASSES[label_id.item()], float(score), [round(v, 3) for v in box.tolist()])

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.imshow(image)
for score, label_id, box in zip(selected_scores, selected_labels, selected_boxes):
    x1, y1, x2, y2 = box_cxcywh_to_xyxy(box.unsqueeze(0))[0].detach().cpu().tolist()
    x1, x2 = x1 * ORIG_W, x2 * ORIG_W
    y1, y2 = y1 * ORIG_H, y2 * ORIG_H
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='orange', facecolor='none')
    ax.add_patch(rect)
    ax.text(x1, max(5, y1 - 8), f"{LEARNING_CLASSES[label_id.item()]} {score.item():.2f}", color='white', bbox=dict(facecolor='orange', alpha=0.8, pad=2))
ax.set_title('Learning Path Inference Demo')
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. 工程路径：加载预训练 DETR

下面开始真正的工程实现。

我们直接使用官方预训练权重：

- `AutoImageProcessor` 负责预处理与后处理
- `DetrForObjectDetection` 负责前向推理

这条路径最接近真实项目中的使用方式。

### 4.3 训练 vs 推理的区别

| 阶段 | 学习路径行为 | 工程路径行为 |
|---|---|---|
| 训练 | `hungarian_match()` 先给 query 分配真值，再计算 `cls + L1 + GIoU` | 训练版 DETR 内部也会做 matcher + criterion |
| 推理 | `softmax -> 去掉 no object -> threshold 过滤` | `post_process_object_detection()` 完成阈值过滤和坐标缩放 |
| 是否需要真值框 | 需要 | 不需要 |
| 是否需要 NMS | 不需要 | 不需要 |

关键点：**训练和推理的区别主要在 supervision 与 post-process，不在网络主体结构。**

## 6. 结果与边界

### 6.1 我们得到了什么

- 学习路径：成功把 backbone、PE、encoder、decoder、prediction heads、matching、loss 串起来了
- 工程路径：成功在真实图像上运行了预训练 DETR，并完成后处理与可视化

### 6.2 两条路径各自的边界

- **学习路径的收益**：透明、可讲、适合面试和原理复现
- **学习路径的边界**：不是一个可直接部署的高精度检测器
- **工程路径的收益**：API 成熟、结果稳定、贴近真实项目
- **工程路径的边界**：很多关键细节被封装起来，不读源码就不容易真正理解

### 6.3 实际建议

- 如果目标是 **准备面试 / 讲论文**，先走学习路径
- 如果目标是 **快速做 demo / 接业务图像**，先走工程路径
- 如果目标是 **真正训练或改 DETR**，则需要进一步进入官方训练代码或更强变体（如 Deformable DETR）

In [ ]:
with torch.no_grad():
    outputs_with_attn = pretrained_model(**inputs, output_attentions=True)

cross_attn = outputs_with_attn.cross_attentions[-1][0].mean(dim=0)
max_probs, max_labels = outputs_with_attn.logits[0].softmax(-1)[:, :-1].max(dim=-1)
top_query_ids = max_probs.argsort(descending=True)[:4]

hw = cross_attn.shape[-1]
h_feat = int(math.sqrt(hw))
w_feat = hw // h_feat
while h_feat > 1 and h_feat * w_feat != hw:
    h_feat -= 1
    w_feat = hw // h_feat

fig, axes = plt.subplots(1, len(top_query_ids), figsize=(4 * len(top_query_ids), 4))
if len(top_query_ids) == 1:
    axes = [axes]

for ax, qid in zip(axes, top_query_ids):
    attn_map = cross_attn[qid].detach().cpu().reshape(h_feat, w_feat)
    label_name = pretrained_model.config.id2label[max_labels[qid].item()]
    ax.imshow(attn_map, cmap='hot')
    ax.set_title(f'Query {qid.item()} {label_name}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. 附录：同一张图上的输出对照

严格说，学习路径里的框并不是训练得到的真实可用检测器输出，因此它和工程路径不能直接比较 AP。

但它们仍然能形成一个很重要的对应关系：

- 学习路径告诉你 `query / matching / loss / filtering` 分别是什么
- 工程路径告诉你这些概念在成熟库里怎样被封装并真正用于推理

这正是“学习实现 vs 工程实现”的核心对照。

In [ ]:
inputs = image_processor(images=image, return_tensors='pt')
inputs = {k: v.to(device) for k, v in inputs.items()}
print({k: tuple(v.shape) for k, v in inputs.items()})

with torch.no_grad():
    outputs = pretrained_model(**inputs)

results = image_processor.post_process_object_detection(
    outputs,
    threshold=0.9,
    target_sizes=torch.tensor([[ORIG_H, ORIG_W]], device=outputs.logits.device),
)[0]

print('detections:', len(results['scores']))
for score, label_id, box in zip(results['scores'], results['labels'], results['boxes']):
    label_name = pretrained_model.config.id2label[label_id.item()]
    box_list = [round(v, 1) for v in box.detach().cpu().tolist()]
    print(f'{label_name:12s} score={score.item():.3f} box={box_list}')

### 4.1 高层 API 背后做了什么

工程路径虽然只写了几行代码，但背后对应的仍然是学习路径里的同一套逻辑：

| 学习路径实现 | 工程库内部对应 | 说明 |
|---|---|---|
| `SimpleBackbone` + `input_proj` | `pretrained_model.model.backbone` | CNN 特征提取与通道投影 |
| `PositionalEncoding2D` | 模型内部位置编码 | 给空间 token 注入位置信息 |
| `EncoderBlock` | encoder layers | 全局建模图像特征 |
| `query_embed` + `DecoderBlock` | decoder + learned object queries | 每个 query 查询一个潜在目标 |
| `PredictionHeads` | `outputs.logits` / `outputs.pred_boxes` | 输出类别与归一化框 |
| `threshold filtering` | `post_process_object_detection()` | 过滤低置信结果并还原框坐标 |

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(9, 6))
ax.imshow(image)
for score, label_id, box in zip(results['scores'], results['labels'], results['boxes']):
    x1, y1, x2, y2 = box.detach().cpu().tolist()
    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor='deepskyblue', facecolor='none')
    ax.add_patch(rect)
    label_name = pretrained_model.config.id2label[label_id.item()]
    ax.text(x1, max(5, y1 - 8), f'{label_name} {score.item():.2f}', color='white', bbox=dict(facecolor='deepskyblue', alpha=0.8, pad=2))
ax.set_title('Engineering Path Detection Results')
ax.axis('off')
plt.tight_layout()
plt.show()

### 4.2 Batch inference 与参数权衡

虽然这里只演示单张图，但工业里通常会关心批量推理。

| 因素 | 增大时 | 吞吐量 | 显存 / 内存 | 速度 | 效果 |
|---|---|---|---|---|---|
| `batch_size` | ↑ | ↑ | ↑↑ | 单样本平均更快 | 基本不变 |
| image resolution | ↑ | ↓ | ↑↑ | ↓ | 可能更好 |
| query 数量 | ↑ | ~ | ↑ | ↓ | 可能略好 |
| threshold | ↑ | 输出更少 | ~ | ~ | 更保守 |
| device: CPU→GPU | ↑ | ↑↑ | GPU 占用上升 | ↑↑↑ | 不变 |

DETR 推理阶段的核心 API 是稳定的，但工程取舍通常在这些参数上。

In [ ]:
batch_images = [image, image.copy()]
batch_inputs = image_processor(images=batch_images, return_tensors='pt')
batch_inputs = {k: v.to(device) for k, v in batch_inputs.items()}

with torch.no_grad():
    batch_outputs = pretrained_model(**batch_inputs)

batch_results = image_processor.post_process_object_detection(
    batch_outputs,
    threshold=0.9,
    target_sizes=torch.tensor([[ORIG_H, ORIG_W], [ORIG_H, ORIG_W]], device=batch_outputs.logits.device),
)

print('batch size:', len(batch_results))
print('detections per image:', [len(item['scores']) for item in batch_results])

## 8. 面试与项目表达

### 高频面试题

**Q1：DETR 为什么不需要 NMS？**

- 训练时不是一对多分配，而是 **Hungarian 一对一匹配**
- 每个真实目标只监督一个 query
- 因此模型天然学习“不同 query 负责不同目标”，推理阶段不再依赖 NMS 去重

**Q2：Object Query 的本质是什么？**

- 它不是 anchor box，也不是图像 patch
- 它是可学习的查询向量
- 可以理解成一组“目标槽位（object slots）”，每个槽位都去图像里找一个潜在目标

**Q3：DETR 为什么收敛慢？**

- query 从零开始学习“该找什么”
- matching 在训练初期不稳定
- encoder 的全局注意力和 decoder 的 cross-attention 都需要较长时间建立有效对齐

**Q4：为什么要同时用 L1 和 GIoU？**

- L1 直接约束坐标数值
- GIoU 直接约束重叠质量
- 两者互补：一个更关注参数差异，一个更关注几何重叠

**Q5：DETR 为什么对小目标不友好？**

- 原版 DETR 主要使用单尺度高层特征
- 小目标在低分辨率特征图上容易丢失细节
- 这也是 Deformable DETR 强调多尺度与稀疏采样的原因

**Q6：学习路径里的 `object query` 和工程路径中的 `config.num_queries` 是什么关系？**

- 本质相同
- 学习路径手写的是 `nn.Embedding(num_queries, d_model)`
- 工程路径里这些 learned queries 被封装在预训练模型内部

**Q7：`post_process_object_detection()` 在做什么？**

- 把模型输出的归一化框映射回原图尺寸
- 根据阈值过滤低置信结果
- 输出 `scores / labels / boxes`，供后续可视化或业务逻辑使用

### 面试速答提纲

| # | 问题 | 一句话回答 |
|---|---|---|
| 1 | DETR 为什么不用 NMS？ | 因为训练时用一对一 Hungarian matching 约束了去重 |
| 2 | Object Query 是什么？ | 可学习的目标查询槽位 |
| 3 | 为什么收敛慢？ | query 对齐和 matching 冷启动都难 |
| 4 | L1 和 GIoU 为什么一起用？ | 同时约束坐标误差与几何重叠 |
| 5 | 工程里怎么快速上手 DETR？ | `AutoImageProcessor + DetrForObjectDetection + post_process_object_detection` |

### 项目表达

> 如果面试官问“你做过什么和 DETR 相关的内容”，可以这样回答：

- **学习深度**：我手写拆解了 DETR 的 backbone、2D 位置编码、encoder、decoder、object query、Hungarian matching 和 set loss
- **工程能力**：我用 HuggingFace 预训练 `facebook/detr-resnet-50` 完成了真实图像的端到端检测、后处理和 attention 可视化
- **对比思考**：我能解释学习实现与工业实现的映射关系，以及为什么 DETR 训练阶段需要 matching、推理阶段不需要 NMS

### 延伸阅读与对比

| 对比维度 | DETR | Deformable DETR | Faster R-CNN |
|---|---|---|---|
| 核心思想 | 集合预测 + query 检测 | 稀疏可变形注意力 + 多尺度 | 候选框 + RoI 二阶段检测 |
| 去重方式 | 训练期 matching | 训练期 matching | 推理期 NMS |
| 小目标能力 | 一般 | 更强 | 通常较强 |
| 工程特征 | 结构简洁 | 收敛更快 | 生态成熟 |

### 进阶探索方向

- Deformable DETR：看它如何用稀疏采样缓解小目标与收敛问题
- DINO / DN-DETR：看 query denoising 如何稳定训练
- DETR segmentation：看 query 框架如何扩展到实例分割 / 全景分割